# Reanalysis code — "Drug identity determines the temporal order of competing adaptive pathways"

This notebook implements the statistical toolkit and data-loading/classification pipeline used to
reanalyse the raw, publicly deposited supplementary datasets from the studies cited in the manuscript.

**Purpose of this notebook.** It is written to accompany the data/code availability statement for
Proceedings B: it is a self-contained, from-source-data rebuild of the described methodology (earliest-
crossing TARGET/REGULATORY classification, Fisher's exact test with Haldane–Anscombe corrected odds
ratios, Holm/Benjamini–Hochberg correction, a Beta-Binomial Bayes factor, ridge-penalised Cox
proportional-hazards regression, and the Nelson–Aalen cumulative-hazard estimator), applied directly to
the original authors' supplementary files.

**Important, honest caveat.** This is an independent rebuild, not a transcription of the original
analysis scripts (those were not available when writing this notebook). Every function has been sanity
checked against synthetic data with known ground truth (see the "self-test" cells), and the Zlamal et
al. (2021) ciprofloxacin pipeline reproduces the *qualitative* pattern reported in the manuscript
(consistent TARGET-first ordering in the *E. coli* / *A. baumannii* runs; a reversal in the *P.
aeruginosa* PAC-1 run). **Please do not assume the exact numbers here match the manuscript's published
tables without checking cell-by-cell against your own working** — in particular, the TARGET/REGULATORY
gene panels below (`GENE_PANELS` in the classification section) were reconstructed from the manuscript
text and should be verified line-by-line against your Methods section, since they are the single most
consequential researcher-degree-of-freedom in the whole pipeline.

**Environment note.** This was built in an offline development environment with no internet access, so `lifelines`
and `statsmodels` were unavailable — the Cox regression and Nelson–Aalen estimator below are therefore
custom `numpy`/`scipy` implementations rather than calls to those packages. If you have installed those
libraries in your own environment, cross-checking against `lifelines.CoxPHFitter` /
`lifelines.NelsonAalenFitter` is strongly recommended before relying on the custom implementations for
publication-facing numbers.

**Data provenance.** All raw data loaded below are the original authors' publicly available
supplementary files (not reproduced in this notebook — only *paths* to files the user has placed
alongside it are referenced). See the Data Availability section for full citations.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats, special, optimize
import re

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

# Point this at the folder containing the raw supplementary files
# (see the Data Availability section at the end of this notebook for the
# full list of source files and where to obtain them).
#
# Works two ways:
#   - Google Colab: mounts your Drive and reads from DRIVE_FOLDER below.
#     Edit DRIVE_FOLDER to match where you uploaded the files, e.g.
#     'MyDrive/reanalysis_pipeline_data' (path is relative to your Drive root,
#     which Colab mounts at /content/drive).
#   - Anywhere else (local Jupyter, etc.): falls back to a local folder;
#     edit UPLOAD_DIR directly in the except branch if needed.
DRIVE_FOLDER = 'MyDrive'  # files are directly in My Drive root

try:
    from google.colab import drive as _gdrive
    _gdrive.mount('/content/drive')
    UPLOAD_DIR = f'/content/drive/{DRIVE_FOLDER}'
except ImportError:
    import os
    # Falls back to a local working directory. Edit this path to point at
    # wherever you have placed the raw supplementary files listed in the
    # Data Availability section below.
    UPLOAD_DIR = os.getcwd()

print('UPLOAD_DIR =', UPLOAD_DIR)


## 1. Statistical toolkit

Reusable, dependency-light implementations of every statistical method described in the manuscript.

In [ ]:
"""
Core statistical toolkit re-implemented from scratch (numpy/scipy/pandas only —
no internet access available in this environment for lifelines/statsmodels).
"""
import numpy as np
import pandas as pd
from scipy import stats, special, optimize

# ---------------------------------------------------------------------------
# 1. Fisher's exact test + Haldane-Anscombe corrected odds ratio & 95% CI
# ---------------------------------------------------------------------------
def fisher_or_ha(a, b, c, d, alpha=0.05):
    """
    2x2 table:
            event      no-event
    grp1      a            b
    grp2      c            d
    Returns Fisher's exact p-value (two-sided) and Haldane-Anscombe corrected
    odds ratio with a Wald 95% CI on the log-OR scale (standard approach when
    any cell is 0, or generally as a conservative default).
    """
    p_value = stats.fisher_exact([[a, b], [c, d]], alternative='two-sided')[1]
    a_c, b_c, c_c, d_c = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    or_ha = (a_c * d_c) / (b_c * c_c)
    se_log_or = np.sqrt(1/a_c + 1/b_c + 1/c_c + 1/d_c)
    z = stats.norm.ppf(1 - alpha/2)
    log_or = np.log(or_ha)
    ci_low = np.exp(log_or - z * se_log_or)
    ci_high = np.exp(log_or + z * se_log_or)
    return {'OR_HA': or_ha, 'CI_low': ci_low, 'CI_high': ci_high, 'p_value': p_value,
            'a': a, 'b': b, 'c': c, 'd': d}


# ---------------------------------------------------------------------------
# 2. Multiple-comparison correction: Holm and Benjamini-Hochberg
# ---------------------------------------------------------------------------
def holm_correction(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = (n - rank) * pvals[idx]
        running_max = max(running_max, val)
        adj[idx] = min(running_max, 1.0)
    return adj

def bh_correction(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(n)
    running_min = 1.0
    for rank in range(n - 1, -1, -1):
        idx = order[rank]
        val = pvals[idx] * n / (rank + 1)
        running_min = min(running_min, val)
        adj[idx] = running_min
    return adj


# ---------------------------------------------------------------------------
# 3. Bayes factor for a 2x2 contingency table (Beta-Binomial, Gunel-Dickey
#    style independent-proportions model vs common-proportion null)
# ---------------------------------------------------------------------------
def bayes_factor_2x2(a, b, c, d, prior_a=1.0, prior_b=1.0):
    """
    BF10 = P(data | H1: p1 != p2) / P(data | H0: p1 == p2)
    H1 places independent Beta(prior_a,prior_b) priors on p1 and p2 separately;
    H0 places a single Beta(prior_a,prior_b) prior on a common p. Binomial
    coefficients C(n1,a)*C(n2,c) are common to both models' marginal
    likelihoods and cancel analytically, leaving a ratio of Beta functions
    (this cancellation was verified numerically against synthetic tables:
    a symmetric table like (5,5,5,5) correctly gives BF10 < 1, and a strongly
    associated table like (9,1,1,9) gives BF10 >> 1).
    a,b = successes/failures in group 1 (n1 = a+b); c,d = group 2 (n2 = c+d).
    """
    n1, n2 = a + b, c + d
    log_bf10 = (special.betaln(a + prior_a, n1 - a + prior_b)
                + special.betaln(c + prior_a, n2 - c + prior_b)
                - special.betaln(prior_a, prior_b)
                - special.betaln(a + c + prior_a, n1 + n2 - a - c + prior_b))
    return float(np.exp(log_bf10))


# ---------------------------------------------------------------------------
# 4. Cox proportional-hazards regression with an L2 (ridge) penalty,
#    Efron's method for ties, fit by Newton-CG on the penalised negative
#    partial log-likelihood.
# ---------------------------------------------------------------------------
def cox_ridge_fit(time, event, X, penalty=1.0, add_intercept=False):
    """
    time    : array of event/censoring times (n,)
    event   : array of 0/1, 1=event observed, 0=censored (n,)
    X       : covariate matrix (n, p) -- should be standardised by the caller
              for the ridge penalty to be meaningful across covariates.
    penalty : ridge penalty strength (lambda); larger = more shrinkage.
    Returns dict with fitted coefficients, standard errors (from the observed
    information at the optimum, unpenalised part only, as an approximation),
    and z / p-values.
    """
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    X = np.asarray(X, dtype=float)
    n, p = X.shape

    order = np.argsort(time)
    time_o, event_o, X_o = time[order], event[order], X[order]

    def neg_log_partial_lik(beta):
        eta = X_o @ beta
        exp_eta = np.exp(eta)
        ll = 0.0
        grad = np.zeros(p)
        # group by unique event times, Efron's method for ties
        unique_times = np.unique(time_o[event_o == 1])
        for t in unique_times:
            risk_idx = np.where(time_o >= t)[0]
            d_idx = np.where((time_o == t) & (event_o == 1))[0]
            d = len(d_idx)
            sum_risk_exp = exp_eta[risk_idx].sum()
            sum_risk_x = (exp_eta[risk_idx, None] * X_o[risk_idx]).sum(axis=0)
            sum_d_eta = eta[d_idx].sum()
            sum_d_x = X_o[d_idx].sum(axis=0)
            sum_d_exp = exp_eta[d_idx].sum()
            sum_d_expx = (exp_eta[d_idx, None] * X_o[d_idx]).sum(axis=0)
            ll += sum_d_eta
            grad += sum_d_x
            for l in range(d):
                denom = sum_risk_exp - (l / d) * sum_d_exp
                numer = sum_risk_x - (l / d) * sum_d_expx
                ll -= np.log(denom)
                grad -= numer / denom
        ll -= penalty * np.sum(beta ** 2)
        grad -= 2 * penalty * beta
        return -ll, -grad

    beta0 = np.zeros(p)
    res = optimize.minimize(neg_log_partial_lik, beta0, jac=True, method='BFGS')
    beta_hat = res.x

    # Approximate covariance via numerical Hessian inverse (unpenalised part
    # dominates for small-to-moderate ridge; treat as an approximation, and
    # always cross-check against a permutation test for small samples).
    eps = 1e-4
    hess = np.zeros((p, p))
    _, g0 = neg_log_partial_lik(beta_hat)
    for j in range(p):
        bp = beta_hat.copy(); bp[j] += eps
        _, gp = neg_log_partial_lik(bp)
        hess[:, j] = (gp - g0) / eps
    hess = (hess + hess.T) / 2
    try:
        cov = np.linalg.inv(hess)
        se = np.sqrt(np.diag(cov))
    except np.linalg.LinAlgError:
        se = np.full(p, np.nan)

    z = beta_hat / se
    pvals = 2 * (1 - stats.norm.cdf(np.abs(z)))
    return {'coef': beta_hat, 'se': se, 'z': z, 'p': pvals, 'hazard_ratio': np.exp(beta_hat)}


# ---------------------------------------------------------------------------
# 5. Nelson-Aalen cumulative hazard estimator (+ Aalen's variance estimator)
# ---------------------------------------------------------------------------
def nelson_aalen(time, event):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    order = np.argsort(time)
    t_sorted, e_sorted = time[order], event[order]
    unique_times = np.unique(t_sorted[e_sorted == 1])
    H = 0.0
    V = 0.0
    rows = []
    for t in unique_times:
        n_at_risk = np.sum(t_sorted >= t)
        d = np.sum((t_sorted == t) & (e_sorted == 1))
        H += d / n_at_risk
        V += d / (n_at_risk ** 2)
        rows.append({'time': t, 'n_risk': n_at_risk, 'n_event': d,
                      'cum_hazard': H, 'cum_hazard_var': V})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 6. Mann-Whitney U (wrapper for genotype-complexity comparisons)
# ---------------------------------------------------------------------------
def mannwhitney(x, y):
    stat, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    return {'U': stat, 'p_value': p, 'n1': len(x), 'n2': len(y)}




### 1.1 Self-tests

Each function is checked against a case with a known/expected answer before being trusted on real data.

In [ ]:
# Fisher / Haldane-Anscombe OR: strong association should give large OR, small p
r = fisher_or_ha(9, 1, 1, 9)
assert r['p_value'] < 0.01 and r['OR_HA'] > 10
print('fisher_or_ha OK:', r)

# Holm / BH: adjusted p-values must be >= raw p-values, and monotonic under sorting
pv = [0.001, 0.02, 0.03, 0.5]
h, b = holm_correction(pv), bh_correction(pv)
assert all(np.array(h) >= np.array(pv) - 1e-12) and all(np.array(b) >= np.array(pv) - 1e-12)
print('Holm:', h, ' BH:', b)

# Bayes factor: a balanced/null-like table should give BF10 < 1; a strongly
# associated table should give BF10 >> 1 (this specific check caught a real
# bug -- an earlier version of this function returned the SAME value for
# both cases due to an incorrect combinatorial normalisation; keep this
# assertion in place if you ever edit bayes_factor_2x2).
bf_null = bayes_factor_2x2(5, 5, 5, 5)
bf_assoc = bayes_factor_2x2(9, 1, 1, 9)
assert bf_null < 1 < bf_assoc, (bf_null, bf_assoc)
print('BF10 (null-like 5,5,5,5):', bf_null, ' BF10 (strong assoc 9,1,1,9):', bf_assoc)

# Cox ridge: recover a known simulated effect size from synthetic survival data
np.random.seed(0)
n = 200
x = np.random.randn(n, 1)
true_beta = 1.2
haz = np.exp(x[:, 0] * true_beta)
t_event = np.random.exponential(1 / haz)
t_cens = np.random.exponential(2, n)
time_obs = np.minimum(t_event, t_cens)
event_obs = (t_event <= t_cens).astype(int)
res = cox_ridge_fit(time_obs, event_obs, x, penalty=0.1)
print('cox_ridge_fit recovered beta (true=1.2):', res['coef'], ' p=', res['p'])
assert 0.5 < res['coef'][0] < 2.0

# Nelson-Aalen: cumulative hazard should be non-decreasing
na = nelson_aalen(time_obs, event_obs)
assert (na['cum_hazard'].diff().dropna() >= 0).all()
print('nelson_aalen OK, final cumulative hazard:', na['cum_hazard'].iloc[-1])
print()
print('All self-tests passed.')


## 2. Data loaders

Each function below converts one source study's idiosyncratic wide-format supplementary spreadsheet
into a common tidy long-format schema (`reactor | time_h | gene | frequency | study | run`), which is
what the classification protocol in Section 3 expects. Column layouts differ subtly between studies
(and even between sheets within the same study — see the Zlamal PAC-1/PAC-2 gene-column-position fix
below), so each loader was written and checked against the actual uploaded file, not against an assumed
generic layout.

In [ ]:
"""
Data loaders for the raw supplementary datasets, converting each source's
idiosyncratic wide-format layout into a common tidy long-format schema:

    reactor_id | time_h | gene | variant | frequency | study | run

This tidy table is the input to the classification protocol in classify.py.
"""
import numpy as np
import pandas as pd
import re

# Reuses UPLOAD_DIR from Section 0 if that cell already ran; otherwise
# resolves it the same Colab-aware way (see Section 0 for details).
if 'UPLOAD_DIR' not in globals():
    DRIVE_FOLDER = 'MyDrive'  # files are directly in My Drive root
    try:
        from google.colab import drive as _gdrive
        _gdrive.mount('/content/drive')
        UPLOAD_DIR = f'/content/drive/{DRIVE_FOLDER}'
    except ImportError:
        import os
        UPLOAD_DIR = os.getcwd()

def _find_header_row(raw, marker_col_idx, marker_text):
    for i in range(len(raw)):
        val = raw.iat[i, marker_col_idx]
        if isinstance(val, str) and marker_text in val:
            return i
    raise ValueError(f'Header marker "{marker_text}" not found in column {marker_col_idx}')


def load_zlamal_run(sheet_name, study='Zlamal2021_mBio', run=None):
    """
    Loads one run (S2A-S2E) of Supplementary Data Set S2 (Zlamal et al. 2021,
    mBio) — reactor x time-point ciprofloxacin evolution SNP frequency data.
    Wide format: row0=title, row1=Reactor N spanners, row2=Day spanners,
    row3=column headers ('Gene name (locus tag)','Variant',<time cols>...,
    'Short annotation','Position/event','Effect type'[, 'Locus Tag']).
    """
    raw = pd.read_excel(f'{UPLOAD_DIR}/mbio.00987-21-sd002.xlsx', sheet_name=sheet_name, header=None)
    header_row = 3
    reactor_row = raw.iloc[1]
    time_row = raw.iloc[header_row]

    # identify gene/variant columns dynamically: CEC/CAB sheets start with
    # 'Gene name (locus tag)' in column 0, but PAC-1/PAC-2 sheets have a
    # leading 'PATRIC (SEED) ID' / 'PATRIC ID' column before 'Gene', so the
    # gene column index is NOT constant across sheets -- must be detected.
    header_labels = [str(v).strip() for v in time_row.tolist()]
    if 'Gene' in header_labels:
        gene_col = header_labels.index('Gene')
    else:
        gene_col = 0  # 'Gene name (locus tag)' sheets: gene is column 0
    variant_col = gene_col + 1
    # trailing metadata columns (Short annotation / Position / Effect type / Locus Tag)
    meta_labels = {'Short annotation', 'Position/event', 'Position', 'Effect type', 'Locus Tag'}
    time_cols = []
    for j in range(2, raw.shape[1]):
        label = time_row.iat[j]
        if isinstance(label, str) and label.strip() in meta_labels:
            continue
        # accept either '76h' (most sheets) or a bare numeric '76' (S2E/PAC-2)
        label_str = str(label).strip()
        numeric_part = label_str[:-1] if label_str.endswith('h') else label_str
        try:
            float(numeric_part)
            time_cols.append(j)
        except (TypeError, ValueError):
            continue

    # forward-fill reactor spanner labels
    reactor_labels = reactor_row.ffill()

    FOOTER_LABELS = {'Sub-total', 'Time hrs', 'IS insert', 'Total'}
    data_start = header_row + 1
    records = []
    for i in range(data_start, len(raw)):
        gene_raw = raw.iat[i, gene_col]
        if isinstance(gene_raw, str) and gene_raw.strip().lower().startswith('preexist'):
            # everything from here on is baseline/ancestral variation present
            # before evolution began -- not part of the acquired-mutation
            # ordering analysis, so stop reading this sheet here.
            break
        if not isinstance(gene_raw, str) or gene_raw.strip() == '' or 'Acquired' in gene_raw:
            continue
        if gene_raw.strip() in FOOTER_LABELS or gene_raw.strip().lower().startswith('sub-total'):
            continue
        gene = re.split(r'\s*\(', gene_raw)[0].strip()
        variant = raw.iat[i, variant_col]
        for j in time_cols:
            freq = raw.iat[i, j]
            if pd.isna(freq):
                continue
            try:
                freq = float(freq)
            except (TypeError, ValueError):
                continue
            reactor = reactor_labels.iat[j]
            time_str = str(time_row.iat[j]).strip()
            time_str = time_str[:-1] if time_str.endswith('h') else time_str
            try:
                time_h = float(time_str)
            except ValueError:
                continue
            records.append({'gene': gene, 'variant': variant, 'reactor': reactor,
                             'time_h': time_h, 'frequency': freq})
    df = pd.DataFrame(records)
    df['study'] = study
    df['run'] = run or sheet_name
    return df


ZLAMAL_RUNS = {
    'S2A(CEC-2)': dict(run='CEC-2', organism='E. coli BW25113'),
    'S2B(CEC-4)': dict(run='CEC-4', organism='E. coli BW25113'),
    'S2C(CAB)':   dict(run='CAB-1', organism='A. baumannii ATCC17978'),
    'S2D(PAC-1)': dict(run='PAC-1', organism='P. aeruginosa ATCC27853'),
    'S2E(PAC-2)': dict(run='PAC-2', organism='P. aeruginosa ATCC27853'),
}

def load_all_zlamal():
    out = []
    for sheet, meta in ZLAMAL_RUNS.items():
        d = load_zlamal_run(sheet, run=meta['run'])
        d['organism'] = meta['organism']
        out.append(d)
    return pd.concat(out, ignore_index=True)


def load_kent_run(sheet_name, run_label, organism='A. baumannii', drug=None):
    """
    Loads one population-level sheet from Kent et al. 2025 (Antimicrob Agents
    Chemother) Table S1/S3-style layout.

    FIXED (this pass, against the now-available aac.00809-25-s0002.xlsx):
    the previous version of this function (a) never derived a `time_h`
    column at all -- classify_reactor() needs one and would fail immediately
    -- and (b) assumed a fixed 'Gene' -> 'Mutation' -> <data> column order
    that holds for S1A/S1B/S3A but is REVERSED in S3B_pop_BAA747_COL (whose
    header row is 'ID','Mutation','Gene',<data...>). Under the old fixed-
    offset assumption this silently dropped reactor 1's t=0 sample in S3B
    (mislabeled as the 'variant' column) rather than raising an error --
    inconsequential here only because t=0 is pre-treatment baseline (always
    0% frequency, never crosses the classification threshold), but not
    something to rely on. This version locates every column (ID/Gene/
    Mutation/sample) by matching its header label directly rather than by
    a fixed offset from 'Gene', so it is robust to either column order.

    Header layout also varies by sheet in a second way: S3A/S3B carry an
    explicit 'Evolution time (hrs):' / 'time (hrs)->' row between the
    Reactor-spanner row and the ID/Gene/Mutation header row; S1A does not
    (no absolute hours given for that arm at all). Where no hour row is
    found, time_h falls back to the ordinal position of each sample's
    trailing letter (A, B, C, ...) within its reactor -- the same
    sample-order fallback the manuscript itself reports using for the
    TGC/ATCC17978 arm specifically (see main text, Table 7 discussion:
    "sample order alone for the TGC/ATCC 17978 arm, which reports no
    absolute sampling times in the original supplementary data").
    """
    raw = pd.read_excel(f'{UPLOAD_DIR}/aac.00809-25-s0002.xlsx', sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(min(10, len(raw))):
        vals = [str(v).strip() for v in raw.iloc[i].tolist()]
        if 'Gene' in vals and 'Mutation' in vals and 'ID' in vals:
            header_row = i
            break
    if header_row is None:
        raise ValueError(f'Could not locate ID/Gene/Mutation header row in sheet {sheet_name}')
    header = raw.iloc[header_row]
    gene_col = list(header).index('Gene')

    reactor_row_idx = None
    for i in range(header_row - 1, -1, -1):
        if any('Reactor' in str(v) for v in raw.iloc[i].tolist()):
            reactor_row_idx = i
            break
    if reactor_row_idx is None:
        raise ValueError(f'Could not locate Reactor-spanner row in sheet {sheet_name}')
    reactor_row = raw.iloc[reactor_row_idx].ffill()

    # explicit hour row, if present, sits between the reactor-spanner row
    # and the ID/Gene/Mutation header row (see docstring)
    hour_row = None
    for i in range(reactor_row_idx + 1, header_row):
        if any('hrs' in str(v).lower() for v in raw.iloc[i].tolist()):
            hour_row = raw.iloc[i]
            break

    # sample columns identified by label pattern (e.g. '1A', '2C'), not by
    # position -- robust to both column orderings described above
    sample_cols = [j for j in range(raw.shape[1])
                   if isinstance(header.iat[j], str) and re.match(r'^\d+[A-Z]$', header.iat[j].strip())]

    records = []
    for i in range(header_row + 1, len(raw)):
        gene = raw.iat[i, gene_col]
        if not isinstance(gene, str) or gene.strip() == '' or gene.strip() in ('Gene', 'TOTAL'):
            continue
        gene = gene.strip()
        for j in sample_cols:
            freq = raw.iat[i, j]
            if pd.isna(freq):
                continue
            try:
                freq = float(freq)
            except (TypeError, ValueError):
                continue
            sample_label = header.iat[j].strip()
            m = re.match(r'^(\d+)([A-Z])$', sample_label)
            time_h = np.nan
            if hour_row is not None:
                try:
                    time_h = float(hour_row.iat[j])
                except (TypeError, ValueError):
                    time_h = np.nan
            records.append({'gene': gene, 'reactor': f'Reactor {m.group(1)}', 'letter': m.group(2),
                             'sample': sample_label, 'time_h': time_h, 'frequency': freq})
    df = pd.DataFrame(records)
    if df['time_h'].isna().all():
        # no absolute-hours row on this sheet -- fall back to ordinal sample
        # order within each reactor (letter position), per the docstring note
        order_map = {c: i for i, c in enumerate(sorted(df['letter'].unique()))}
        df['time_h'] = df['letter'].map(order_map).astype(float)
    df['study'] = 'Kent2025_AAC'
    df['run'] = run_label
    df['organism'] = organism
    df['drug'] = drug
    return df



def load_leyn_gp6(filepath_sheet_pairs):
    """
    Loyn et al. GP6 dataset (media-1.xlsx sheets S1A-C, media-2.xlsx S2A-C).
    Same layout family as Zlamal (gene/variant + reactor-spanner + time header).
    filepath_sheet_pairs: list of (filepath, sheet_name, run_label) tuples.
    """
    out = []
    for filepath, sheet_name, run_label in filepath_sheet_pairs:
        raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
        header_row = None
        for i in range(min(10, len(raw))):
            row_vals = [str(v) for v in raw.iloc[i].tolist()]
            if any(v.strip() == 'Gene' for v in row_vals):
                header_row = i
                break
        if header_row is None:
            continue
        header = raw.iloc[header_row]
        reactor_row = raw.iloc[header_row - 2].ffill() if header_row >= 2 else None
        time_row = raw.iloc[header_row - 1] if header_row >= 1 else None
        gene_col = list(header).index('Gene')
        variant_col = gene_col + 1
        meta_labels = {'Effect', 'Locus Tag', 'Chrom', 'Pos', 'Ref', 'Alt'}
        time_cols = []
        for j in range(gene_col + 2, raw.shape[1]):
            label = header.iat[j]
            if isinstance(label, str) and label.strip() in meta_labels:
                continue
            col_vals = pd.to_numeric(raw.iloc[header_row+1:, j], errors='coerce')
            if col_vals.notna().mean() > 0.3:
                time_cols.append(j)
        variant_col_for_mut = gene_col + 1  # 'Mutation' column
        for i in range(header_row + 1, len(raw)):
            gene = raw.iat[i, gene_col]
            if not isinstance(gene, str) or gene.strip() == '':
                continue
            mutation_label = raw.iat[i, variant_col_for_mut]
            # Skip CNV/coverage rows (Mutation == '-'): verified against the raw
            # sheet that these are a large contiguous block of ~55 genes (incl.
            # mdtK) with values already >50% at t=0 and exceeding 1.0 later --
            # a read-depth/copy-number ratio, not a 0-1 allele frequency, and
            # not compatible with the >=5%-frequency earliest-crossing rule.
            # Including them wrongly forced REG_FIRST at t=0 in every GP6
            # reactor (mdtK sits inside this duplicated genomic region).
            if isinstance(mutation_label, str) and mutation_label.strip() == '-':
                continue
            gene_clean = re.split(r'<>|\s', gene.strip())[0]
            for j in time_cols:
                freq = raw.iat[i, j]
                if pd.isna(freq):
                    continue
                try:
                    freq = float(freq)
                except (TypeError, ValueError):
                    continue
                reactor = reactor_row.iat[j] if reactor_row is not None else np.nan
                time_label = str(time_row.iat[j]).replace('hrs', '').replace('h', '').strip() if time_row is not None else None
                try:
                    time_h = float(time_label)
                except (TypeError, ValueError):
                    time_h = np.nan
                out.append({'gene': gene_clean, 'reactor': reactor, 'sample_col': header.iat[j],
                            'time_h': time_h, 'frequency': freq, 'study': 'Leyn2024_NPJ', 'run': run_label})
    return pd.DataFrame(out)


def load_lindsey_table2():
    """
    Lindsey et al. 2013 (Nature) Supplementary Table 2 — full list of rpoB
    mutations from sequenced isolates, transcribed directly from the
    supplementary PDF text (41586_2013_BFnature11879_MOESM27_ESM.pdf), since
    the PDF stores this as a text table rather than a machine-readable sheet.
    Columns: Treatment, Population, Mutation_location, Ancestral_base,
    Mutant_base, First_to_fix, Residue_location, Ancestral_aa, Mutant_aa.
    """
    # transcribed verbatim from the supplementary PDF (Supplementary Table 2)
    rows = [
        ('Gradual',13,428,'G','A','No',143,'Arginine','Histidine'),
        ('Gradual',13,1721,'C','T','Yes',574,'Serine','Phenylalanine'),
        ('Gradual',14,1691,'C','T','Yes',564,'Proline','Leucine'),
        ('Gradual',17,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Gradual',17,1687,'A','C','Yes',563,'Threonine','Proline'),
        ('Gradual',18,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Gradual',18,1527,'C','A','Yes',509,'Serine','Arginine'),
        ('Gradual',35,443,'A','T','Yes',148,'Glutamine','Leucine'),
        ('Gradual',35,1534,'T','C','No',512,'Serine','Proline'),
        ('Gradual',41,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Gradual',41,1714,'A','C','Yes',572,'Isoleucine','Leucine'),
        ('Gradual',42,443,'A','C','Yes',148,'Glutamine','Proline'),
        ('Gradual',42,1525,'A','C','No',509,'Serine','Arginine'),
        ('Gradual',46,443,'A','T','Yes',148,'Glutamine','Leucine'),
        ('Gradual',46,1715,'T','G','No',572,'Isoleucine','Serine'),
        ('Gradual',49,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Gradual',49,1532,'T','C','Yes',511,'Leucine','Proline'),
        ('Gradual',49,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Gradual',66,1547,'A','G','Yes',516,'Aspartic Acid','Glycine'),
        ('Gradual',69,1534,'T','C','No',512,'Serine','Proline'),
        ('Gradual',69,1597,'C','T','Yes',533,'Leucine','Phenylalanine'),
        ('Gradual',70,1714,'A','C','No',572,'Isoleucine','Leucine'),
        ('Gradual',70,1715,'T','A','Yes',572,'Isoleucine','Asparagine'),
        ('Gradual',71,1585,'C','T','Yes',529,'Arginine','Cysteine'),
        ('Gradual',77,1703,'A','G','No',568,'Asparagine','Serine'),
        ('Gradual',77,1721,'C','T','Yes',574,'Serine','Phenylalanine'),
        ('Gradual',94,1535,'C','T','Yes',512,'Serine','Phenylalanine'),
        ('Gradual',95,1535,'C','T','Yes',512,'Serine','Phenylalanine'),
        ('Gradual',96,427,'C','T','No',143,'Arginine','Cysteine'),
        ('Gradual',96,1547,'A','G','Yes',516,'Aspartic Acid','Glycine'),
        ('Gradual',99,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Gradual',112,1527,'C','A','Yes',509,'Serine','Arginine'),
        ('Gradual',112,1577,'A','G','No',526,'Histidine','Arginine'),
        ('Gradual',119,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Gradual',119,1715,'T','A','Yes',572,'Isoleucine','Asparagine'),
        ('Gradual',123,1546,'G','A','Yes',516,'Aspartic Acid','Asparagine'),
        ('Gradual',126,1538,'A','G','No',513,'Glutamine','Arginine'),
        ('Gradual',126,1601,'G','A','Yes',534,'Glycine','Aspartic Acid'),
        ('Gradual',131,1527,'C','A','No',509,'Serine','Arginine'),
        ('Gradual',131,1687,'A','C','Yes',563,'Threonine','Proline'),
        ('Gradual',135,1534,'T','C','No',512,'Serine','Proline'),
        ('Gradual',135,1545,'G','A','Yes',515,'Methionine','Isoleucine'),
        ('Gradual',140,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Gradual',141,1532,'T','C','Yes',511,'Leucine','Proline'),
        ('Gradual',141,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Gradual',143,1714,'A','C','Yes',572,'Isoleucine','Leucine'),
        ('Gradual',165,1552,'A','T','No',518,'Asparagine','Tyrosine'),
        ('Gradual',165,1715,'T','G','Yes',572,'Isoleucine','Serine'),
        ('Gradual',179,1534,'T','C','Yes',512,'Serine','Proline'),
        ('Gradual',183,1538,'A','C','Yes',513,'Glutamine','Proline'),
        ('Gradual',183,1690,'C','T','No',564,'Proline','Serine'),
        ('Moderate',185,1532,'T','G','Yes',511,'Leucine','Arginine'),
        ('Moderate',185,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Moderate',190,443,'A','T','Yes',148,'Glutamine','Leucine'),
        ('Moderate',190,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Moderate',196,1547,'A','G','No',516,'Aspartic Acid','Glycine'),
        ('Moderate',196,1687,'A','C','Yes',563,'Threonine','Proline'),
        ('Moderate',217,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Moderate',242,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Moderate',242,1715,'T','A','Yes',572,'Isoleucine','Asparagine'),
        ('Moderate',244,1513,'T','C','No',505,'Phenylalanine','Leucine'),
        ('Moderate',244,1534,'T','C','Yes',512,'Serine','Proline'),
        ('Moderate',250,1532,'T','A','Yes',511,'Leucine','Glutamine'),
        ('Moderate',250,1538,'A','T','No',513,'Glutamine','Leucine'),
        ('Moderate',275,1546,'G','A','Yes',516,'Aspartic Acid','Asparagine'),
        ('Moderate',281,443,'A','T','No',148,'Glutamine','Leucine'),
        ('Moderate',281,1714,'A','C','Yes',572,'Isoleucine','Leucine'),
        ('Moderate',307,1547,'A','G','No',516,'Aspartic Acid','Glycine'),
        ('Moderate',307,1589,'T','C','Yes',530,'Isoleucine','Threonine'),
        ('Moderate',315,437,'T','C','No',146,'Valine','Alanine'),
        ('Moderate',315,1535,'C','T','Yes',512,'Serine','Phenylalanine'),
        ('Moderate',318,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Moderate',326,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Moderate',326,1715,'T','A','No',572,'Isoleucine','Asparagine'),
        ('Moderate',357,1534,'T','C','Yes',512,'Serine','Proline'),
        ('Moderate',358,437,'T','A','Yes',146,'Valine','Aspartic Acid'),
        ('Moderate',358,1685,'A','C','No',562,'Glutamic Acid','Alanine'),
        ('Moderate',362,1546,'G','A','Yes',516,'Aspartic Acid','Asparagine'),
        ('Moderate',367,1532,'T','G','Yes',511,'Leucine','Arginine'),
        ('Moderate',367,1546,'G','A','No',516,'Aspartic Acid','Asparagine'),
        ('Moderate',368,1714,'A','C','Yes',572,'Isoleucine','Leucine'),
        ('Moderate',368,1721,'C','T','No',574,'Serine','Phenylalanine'),
        ('Moderate',377,1592,'C','T','No',531,'Serine','Phenylalanine'),
        ('Moderate',377,1600,'G','C','Yes',534,'Glycine','Arginine'),
        ('Moderate',382,1547,'A','G','No',516,'Aspartic Acid','Glycine'),
        ('Moderate',382,1601,'G','T','Yes',534,'Glycine','Valine'),
        ('Moderate',385,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Moderate',387,1547,'A','G','Yes',516,'Aspartic Acid','Glycine'),
        ('Moderate',387,1565,'C','T','No',522,'Serine','Phenylalanine'),
        ('Moderate',389,1703,'A','G','No',568,'Asparagine','Serine'),
        ('Moderate',389,1715,'T','G','Yes',572,'Isoleucine','Serine'),
        ('Moderate',408,80,'T','C','No',27,'Leucine','Proline'),
        ('Moderate',408,1600,'G','T','Yes',534,'Glycine','Cysteine'),
        ('Moderate',408,1703,'A','G','No',568,'Asparagine','Serine'),
        ('Moderate',424,1547,'A','G','Yes',516,'Aspartic Acid','Glycine'),
        ('Moderate',424,1703,'A','G','No',568,'Asparagine','Serine'),
        ('Moderate',433,1547,'A','G','No',516,'Aspartic Acid','Glycine'),
        ('Moderate',433,1703,'A','G','No',568,'Asparagine','Serine'),
        ('Moderate',433,2059,'C','T','Yes',687,'Arginine','Cysteine'),
        ('Moderate',434,443,'A','T','Yes',148,'Glutamine','Leucine'),
        ('Moderate',434,1538,'A','C','No',513,'Glutamine','Proline'),
        ('Moderate',434,1572,'T','G','No',524,'Isoleucine','Methionine'),
        ('Moderate',437,443,'A','T','Yes',148,'Glutamine','Leucine'),
        ('Moderate',437,1535,'C','T','No',512,'Serine','Phenylalanine'),
        ('Moderate',443,437,'T','C','No',146,'Valine','Alanine'),
        ('Moderate',443,1715,'T','G','Yes',572,'Isoleucine','Serine'),
        ('Moderate',459,1527,'C','A','Yes',509,'Serine','Arginine'),
        ('Moderate',459,1610,'G','A','No',537,'Glycine','Aspartic Acid'),
        ('Sudden',477,1714,'A','T','Yes',572,'Isoleucine','Phenylalanine'),
        ('Sudden',497,1546,'G','T','Yes',516,'Aspartic Acid','Tyrosine'),
        ('Sudden',504,1592,'C','A','Yes',531,'Serine','Tyrosine'),
        ('Sudden',513,1546,'G','T','Yes',516,'Aspartic Acid','Tyrosine'),
        ('Sudden',582,1546,'G','T','Yes',516,'Aspartic Acid','Tyrosine'),
        ('Sudden',593,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Sudden',742,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Sudden',762,1592,'C','A','Yes',531,'Serine','Tyrosine'),
        ('Sudden',767,1714,'A','T','Yes',572,'Isoleucine','Phenylalanine'),
        ('Sudden',878,1592,'C','A','Yes',531,'Serine','Tyrosine'),
        ('Sudden',1223,1592,'C','T','Yes',531,'Serine','Phenylalanine'),
        ('Sudden',1236,436,'G','T','Yes',146,'Valine','Phenylalanine'),
        ('Sudden',1334,436,'G','T','Yes',146,'Valine','Phenylalanine'),
    ]
    df = pd.DataFrame(rows, columns=['treatment','population','position','ancestral_base',
                                      'mutant_base','first_to_fix','residue','ancestral_aa','mutant_aa'])
    return df


def load_leyn_triclosan_clones():
    """Curated clone-level genotype/MIC tables for the Leyn triclosan dataset
    (already tidied by the author into CSV form from Data Set S6 of the
    original supplementary Excel file)."""
    linked = pd.read_csv(f'{UPLOAD_DIR}/S6_cadC_fabI_linked_clones.csv')
    full = pd.read_csv(f'{UPLOAD_DIR}/S6_clone_genotypes_MIC.csv')
    return linked, full




def load_zlamal_clone_complexity(sheet_name, gene_col_name='gene', section_marker_col=0):
    """
    Loads a Data Set S3 clone-level sheet (S3A/S3B/S3C) and returns, per clone
    (column), the number of '+' (mutation-present) marks -- a simple proxy
    for genotype complexity / number of mutations per evolved clone, usable
    with mannwhitney() to compare complexity between runs (as in ESM Table
    S1's D_eff-style comparison). Only the two header rows (gene/variant,
    RUN=) plus data rows are used; section marker rows (e.g. 'A. Potential
    driver mutations') are skipped.
    """
    raw = pd.read_excel(f'{UPLOAD_DIR}/mbio.00987-21-sd003.xlsx', sheet_name=sheet_name, header=None)
    run_row = raw.iloc[1]
    header = raw.iloc[2]
    gene_col = list(header).index('gene') if 'gene' in list(header) else 0
    variant_col = gene_col + 1
    clone_cols = []
    for j in range(variant_col + 1, raw.shape[1]):
        label = header.iat[j]
        if isinstance(label, str) and label.strip() not in ('', 'codon_mutation', 'new_seq',
                                                              'repeat_mutation', 'mobile_element',
                                                              'indel_size', 'locus_tag'):
            clone_cols.append(j)
        else:
            break  # stop at first trailing-metadata column
    counts = {}
    runs = {}
    for j in clone_cols:
        clone_id = header.iat[j]
        run_label = run_row.iat[j]
        n_mut = 0
        for i in range(3, len(raw)):
            val = raw.iat[i, j]
            if isinstance(val, str) and val.strip() == '+':
                n_mut += 1
        counts[clone_id] = n_mut
        runs[clone_id] = run_label
    out = pd.DataFrame({'clone': list(counts.keys()),
                         'n_mutations': list(counts.values()),
                         'run': [runs[c] for c in counts.keys()]})
    return out


## 3. Classification protocol

In [ ]:
"""
Classification protocol: for each (run, reactor), determine whether the
evolutionary trajectory is TARGET-first, REGULATION-first, co-emergent
(both cross the threshold at the same sampled time point), or neither
(neither gene class ever reaches the threshold), based on the earliest
sampled time point at which any locus in each gene class first crosses a
detection-frequency threshold (default 5%), matching the "earliest-crossing"
rule described in the manuscript's classification protocol.

NOTE ON GENE PANELS: TARGET/REGULATORY gene-class assignments below follow
the drug/organism logic described in the manuscript (direct antibiotic
target vs. efflux/regulatory adaptation). These panels are reconstructed
from the manuscript text and the structure of each supplementary dataset;
please cross-check them line-by-line against your own Methods section
before treating any downstream number as final -- gene panel definitions
are the single most consequential researcher-degree-of-freedom in this
whole pipeline.
"""
import pandas as pd
import numpy as np

GENE_PANELS = {
    # (study, run) -> {'TARGET': {...}, 'REG': {...}}
    ('Zlamal2021_mBio', 'CEC-2'): {'TARGET': {'gyrA', 'gyrB', 'parC', 'parE'}, 'REG': {'marR', 'soxR', 'acrR'}},
    ('Zlamal2021_mBio', 'CEC-4'): {'TARGET': {'gyrA', 'gyrB', 'parC', 'parE'}, 'REG': {'marR', 'soxR', 'acrR'}},
    ('Zlamal2021_mBio', 'CAB-1'): {'TARGET': {'gyrA', 'gyrB', 'parC', 'parE'}, 'REG': {'adeN', 'adeS'}},
    # 'adeR' removed above: the manuscript's Table 1 protocol text lists REG
    # for A. baumannii as {adeN, adeS} only (adeR belongs to the separate
    # Kent et al. TGC panel). Checked against the raw CAB-1 sheet: adeR never
    # appears as a mutated gene here, so this is a definition fix, not a
    # numeric one.
    ('Zlamal2021_mBio', 'PAC-1'): {'TARGET': {'gyrA', 'gyrB'}, 'REG': {'nfxB', 'mexS', 'mexT'}},
    ('Zlamal2021_mBio', 'PAC-2'): {'TARGET': {'gyrA', 'gyrB'}, 'REG': {'nfxB', 'mexS', 'mexT'}},
    # GP6 (Leyn et al. 2024, NPJ Antimicrob Resist), panel exactly as stated
    # in the manuscript text for Table 5.
    ('Leyn2024_NPJ', 'GP6_Ecoli'): {'TARGET': {'gyrA', 'gyrB', 'parC', 'parE'}, 'REG': {'acrR', 'marR', 'mdtK'}},
    ('Leyn2024_NPJ', 'GP6_Abaumannii'): {'TARGET': {'gyrA', 'gyrB', 'parC', 'parE'}, 'REG': {'adeN', 'adeS', 'mdtK'}},
    # Yu et al. 2025 (Microbiol Spectr), fleroxacin dose-response; panel and
    # 10% threshold as stated in ESM Table S2 (note: different threshold
    # from the 5% used everywhere else -- pass threshold=0.10 explicitly
    # when classifying this dataset, see Section 10 below).
    ('Yu2025_MicrobiolSpectr', 'low-dose'): {'TARGET': {'gyrA', 'gyrB'}, 'REG': {'nfxB', 'mexR', 'nalC', 'nalD'}},
    ('Yu2025_MicrobiolSpectr', 'high-dose'): {'TARGET': {'gyrA', 'gyrB'}, 'REG': {'nfxB', 'mexR', 'nalC', 'nalD'}},
    # Kent et al. 2025 (Antimicrob Agents Chemother), COL/TGC in A. baumannii
    # (Table 6). Panel is drug-specific, not strain-specific -- same TARGET/REG
    # sets apply to both ATCC17978 and BAA-747 runs of a given drug. TGC REG
    # set includes the two cis-regulatory insertion events (us_adeA, us_abeM)
    # that produce the same adeAB(C)-overexpression phenotype as trans-acting
    # adeSR mutations, per the manuscript's stated protocol; 'trm' (a third,
    # unclassified mutational class discussed in Limitations) is deliberately
    # excluded from both TARGET and REG.
    ('Kent2025_AAC', 'TGC'): {'TARGET': {'rpsJ'}, 'REG': {'adeR', 'adeS', 'adeN', 'us_adeA', 'us_abeM'}},
    ('Kent2025_AAC', 'COL'): {'TARGET': {'lpxA', 'lpxC', 'lpxD'}, 'REG': {'pmrA', 'pmrB', 'pmrC'}},
}

def classify_reactor(df_run, threshold=0.05, time_col='time_h', reactor_col='reactor',
                      gene_col='gene', freq_col='frequency', target_genes=None, reg_genes=None):
    """
    df_run: tidy long-format dataframe already filtered to a single run.
    Returns a per-reactor classification dataframe.
    """
    results = []
    # Exclude pooled per-gene aggregate rows (variant == 'all', present only in
    # the Zlamal PAC-1/PAC-2 sheets for a few genes e.g. nfxB, mexS). The
    # manuscript's protocol specifies "any TARGET/REG-pathway VARIANT reached
    # >=threshold", i.e. a single named mutation crossing the line, not the
    # summed frequency of several individually sub-threshold variants at the
    # same locus. Verified against the raw Data Set S2 values: including the
    # 'all' rows wrongly called REG-pathway establishment in PAC-2 Reactor 4
    # at 26h from a pooled nfxB frequency of 6.9%, when no single nfxB variant
    # exceeded 4.3% until 76h -- this reactor is TARGET-first once the
    # aggregate row is excluded, matching the manuscript's Table 1 (3/6
    # TARGET-first, 1/6 REG-first, 2/6 co-emergent for PAC-2; the unfixed
    # code gave 2/3/1, an extra spurious co-emergent call).
    if 'variant' in df_run.columns:
        df_run = df_run[df_run['variant'] != 'all']
    for reactor, g in df_run.groupby(reactor_col):
        g_target = g[g[gene_col].isin(target_genes)]
        g_reg = g[g[gene_col].isin(reg_genes)]

        t_target = g_target.loc[g_target[freq_col] >= threshold, time_col]
        t_reg = g_reg.loc[g_reg[freq_col] >= threshold, time_col]

        first_target = t_target.min() if len(t_target) else np.nan
        first_reg = t_reg.min() if len(t_reg) else np.nan

        if np.isnan(first_target) and np.isnan(first_reg):
            ordering = 'NEITHER'
        elif np.isnan(first_reg) or (not np.isnan(first_target) and first_target < first_reg):
            ordering = 'TARGET_FIRST'
        elif np.isnan(first_target) or (first_reg < first_target):
            ordering = 'REG_FIRST'
        else:
            ordering = 'CO_EMERGENT'

        results.append({
            'reactor': reactor, 'first_target_crossing_h': first_target,
            'first_reg_crossing_h': first_reg, 'ordering': ordering,
        })
    return pd.DataFrame(results)


def classify_all_zlamal(df, threshold=0.05):
    out = []
    for run, g in df.groupby('run'):
        panel = GENE_PANELS[('Zlamal2021_mBio', run)]
        res = classify_reactor(g, threshold=threshold, target_genes=panel['TARGET'], reg_genes=panel['REG'])
        res['run'] = run
        out.append(res)
    return pd.concat(out, ignore_index=True)


def build_2x2_from_classification(class_df, group_col='run', positive_group=None, negative_groups=None):
    """
    Build a 2x2 contingency table (TARGET_FIRST vs REG_FIRST) comparing one
    run/group against a pooled set of other runs/groups, for use with
    fisher_or_ha() / bayes_factor_2x2(). CO_EMERGENT / NEITHER reactors are
    excluded from this particular 2x2 (as in a standard ordering comparison).
    """
    sub = class_df[class_df['ordering'].isin(['TARGET_FIRST', 'REG_FIRST'])]
    pos = sub[sub[group_col] == positive_group]
    neg = sub[sub[group_col].isin(negative_groups)]
    a = (pos['ordering'] == 'TARGET_FIRST').sum()
    b = (pos['ordering'] == 'REG_FIRST').sum()
    c = (neg['ordering'] == 'TARGET_FIRST').sum()
    d = (neg['ordering'] == 'REG_FIRST').sum()
    return a, b, c, d


def build_2x2_reg_vs_not(class_df_a, class_df_b):
    """
    Build the 2x2 table actually used for the manuscript's drug-identity and
    escalation-regimen comparisons (Table 5; PAC-1 vs PAC-2): REG-first-or-
    REG-only vs everything else (TARGET-first + co-emergent + neither),
    using the FULL reactor count as the denominator for each group -- unlike
    build_2x2_from_classification() above, co-emergent reactors are *not*
    excluded here. This was verified against the manuscript's own reported
    numbers: it is the only framing that reproduces Table 5's stated p =
    0.0082 / OR = 37.8 (E. coli, CIP vs GP6), p = 0.0152 / OR = 40.3
    (A. baumannii, CIP vs GP6), and the PAC-1-vs-PAC-2 p = 0.040 (one-
    tailed) / 0.080 (two-tailed) -- the build_2x2_from_classification()
    TARGET-vs-REG-excluding-co-emergent framing does NOT reproduce these
    numbers and should not be used for these two comparisons.
    class_df_a, class_df_b: classification dataframes (e.g. one run, or
    several runs concatenated) for the two groups being compared.
    """
    a = (class_df_a['ordering'] == 'REG_FIRST').sum()
    b = len(class_df_a) - a
    c = (class_df_b['ordering'] == 'REG_FIRST').sum()
    d = len(class_df_b) - c
    return a, b, c, d




## 4. Primary analysis: ciprofloxacin evolution (Zlamal et al. 2021, *mBio*)

Loads all five morbidostat runs (Data Set S2: CEC-2, CEC-4, CAB-1, PAC-1, PAC-2), classifies each
reactor's trajectory, and reproduces the manuscript's core drug-identity comparison.

In [ ]:
zlamal_df = load_all_zlamal()
print('Loaded', len(zlamal_df), 'tidy (gene, reactor, time_h, frequency) rows across', zlamal_df['run'].nunique(), 'runs')
zlamal_df.groupby('run').agg(n_reactors=('reactor', 'nunique'), n_rows=('gene', 'size'))


In [ ]:
zlamal_class = classify_all_zlamal(zlamal_df, threshold=0.05)
print(zlamal_class.sort_values(['run', 'reactor']).to_string(index=False))
print()
print('Ordering counts per run:')
print(zlamal_class.groupby(['run', 'ordering']).size().unstack(fill_value=0))


### 4.1 Drug-identity comparison

*E. coli* / *A. baumannii* ciprofloxacin runs (CEC-2, CEC-4, CAB-1) vs. the *P. aeruginosa* runs
(PAC-1, PAC-2), using the same Fisher's-exact + Haldane–Anscombe-OR + Bayes-factor toolkit described in
the manuscript. CO_EMERGENT / NEITHER reactors are excluded from this particular 2×2, as in a standard
TARGET-first-vs-REG-first ordering comparison.

In [ ]:
# explicit pooled 2x2 comparison (clearer here than the single-vs-pooled-rest
# helper build_2x2_from_classification, which is provided in Section 3 for
# the common case of "one run vs. all others pooled")
grp_ecoli_ab = zlamal_class[zlamal_class['run'].isin(['CEC-2', 'CEC-4', 'CAB-1'])]
grp_ecoli_ab = grp_ecoli_ab[grp_ecoli_ab['ordering'].isin(['TARGET_FIRST', 'REG_FIRST'])]
grp_pae = zlamal_class[zlamal_class['run'].isin(['PAC-1', 'PAC-2'])]
grp_pae = grp_pae[grp_pae['ordering'].isin(['TARGET_FIRST', 'REG_FIRST'])]

a = (grp_ecoli_ab['ordering'] == 'TARGET_FIRST').sum()
b = (grp_ecoli_ab['ordering'] == 'REG_FIRST').sum()
c = (grp_pae['ordering'] == 'TARGET_FIRST').sum()
d = (grp_pae['ordering'] == 'REG_FIRST').sum()
print(f'2x2 table -- TARGET_FIRST / REG_FIRST:  E.coli+A.baumannii=({a},{b})  P.aeruginosa=({c},{d})')

result = fisher_or_ha(a, b, c, d)
print('Fisher / Haldane-Anscombe OR:', result)
print('Bayes factor (BF10):', bayes_factor_2x2(a, b, c, d))


### 4.2 Secondary, exploratory comparison: PAC-1 vs. PAC-2 escalation regimens

The manuscript treats this comparison as underpowered and tentative (~38% power, does not survive
multiple-comparison correction) — reproduced here for transparency, not as a robust finding.

In [ ]:
# NOTE (fixed): the manuscript's actual PAC-1-vs-PAC-2 test is REG-first-or-
# not (full n=6 denominator per run, co-emergent counted as "not"), not a
# TARGET-vs-REG comparison excluding co-emergent reactors -- see the
# docstring of build_2x2_reg_vs_not() in Section 3 for how this was
# verified (reproduces the manuscript's p = 0.040 one-tailed / 0.080
# two-tailed exactly; the excluding-co-emergent framing does not).
pac1 = zlamal_class[zlamal_class['run'] == 'PAC-1']
pac2 = zlamal_class[zlamal_class['run'] == 'PAC-2']
a2, b2, c2, d2 = build_2x2_reg_vs_not(pac1, pac2)
print(f'PAC-1 REG-first: {a2}/{len(pac1)}   PAC-2 REG-first: {c2}/{len(pac2)}')
print(fisher_or_ha(a2, b2, c2, d2))
from scipy import stats as _st
print('one-tailed (PAC-1 REG-enriched):', _st.fisher_exact([[a2, b2], [c2, d2]], alternative='greater')[1])
print('n reactors in each comparison (informal power check -- consistent with the manuscript treating')
print('this as underpowered):', len(pac1), 'vs', len(pac2))

### 4.3 Pooled time-to-target-mutation: ridge Cox regression + Nelson–Aalen

Treats "first sampled time at which a TARGET-class mutation crosses 5%" as an event time (censored for
reactors that never cross), with run identity as covariates -- analogous in spirit to the manuscript's
pooled Cox model (Table 8), though the exact time-normalisation choices there should be checked against
your own Methods description before treating these coefficients as final.

In [ ]:
cox_input = zlamal_class.copy()
cox_input['event'] = cox_input['first_target_crossing_h'].notna().astype(int)
# right-censor at the last observed sampling time per run for reactors that never crossed
max_time_per_run = zlamal_df.groupby('run')['time_h'].max()
cox_input['time_h'] = cox_input.apply(
    lambda r: r['first_target_crossing_h'] if r['event'] == 1 else max_time_per_run[r['run']], axis=1)

X = pd.get_dummies(cox_input['run'], drop_first=True).astype(float)
cox_res = cox_ridge_fit(cox_input['time_h'].values, cox_input['event'].values, X.values, penalty=0.5)
coef_table = pd.DataFrame({'run_vs_CAB-1_reference': X.columns, 'coef': cox_res['coef'],
                            'hazard_ratio': cox_res['hazard_ratio'], 'p': cox_res['p']})
print(coef_table)

na_table = nelson_aalen(cox_input['time_h'].values, cox_input['event'].values)
na_table


## 5. Genotype complexity (clone-level data, Data Set S3)

Counts mutations per sequenced clone (a simple proxy in the spirit of a participation-ratio /
complexity measure) and compares runs with `mannwhitney()`.

In [ ]:
complexity_cec = load_zlamal_clone_complexity('S3A  (CEC-2,4)')
print(complexity_cec.groupby('run')['n_mutations'].describe())
cec2_vals = complexity_cec.loc[complexity_cec['run'] == 'CEC2', 'n_mutations'].values
cec4_vals = complexity_cec.loc[complexity_cec['run'] == 'CEC4', 'n_mutations'].values
print(mannwhitney(cec2_vals, cec4_vals))


## 6. Colistin / tigecycline in *A. baumannii* (Kent et al. 2025, *Antimicrob Agents Chemother*)

`aac.00809-25-s0002.xlsx` is a large multi-sheet file (population time-series, qPCR, clone MICs, and
per-regulator statistical summaries already computed by the original authors in sheets `Table S7`–`S14`).
The loader below handles the population time-series sheets (`S1*_pop_*`, `S3*_pop_*`); the pre-computed
summary sheets are best read directly with `pandas.read_excel` since they are already tabular results
rather than raw time series.
**Update:** `aac.00809-25-s0002.xlsx` was not available in the previous session (loader raised
`FileNotFoundError`); it has now been supplied and this section reproduces Table 6 in full.
Getting there required two fixes to the loader itself, found by checking its output against the raw
sheets rather than trusting a successful run:

1. **It never derived a `time_h` column at all.** `classify_reactor()` requires one, so the loader
   was silently incomplete for this study regardless of file availability.
2. **A fixed-offset column assumption breaks on one sheet.** `S1A_pop_17978_TGC`, `S1B_pop_BAA747_TGC`,
   and `S3A_pop_17978_COL` all use header order `ID, Gene, Mutation, <samples...>`, but
   `S3B_pop_BAA747_COL` reverses it to `ID, Mutation, Gene, <samples...>`. The old loader located
   `Gene` and then assumed `variant_col = gene_col + 1` and `first_sample_col = gene_col + 2`; on
   S3B this silently mislabels reactor 1's t=0 sample as the `variant` column and drops it from the
   data instead of raising an error. It happens to be inconsequential here (t=0 is pre-treatment
   baseline, always 0% frequency, so it can never be an earliest-crossing sample) but is not something
   to rely on. The loader below locates every column (`ID`/`Gene`/`Mutation`/sample) by matching its
   header label directly, so it no longer depends on column order.

There is a third, more minor layout difference — only `S3A`/`S3B` carry an explicit
`'Evolution time (hrs):'` row; `S1A` does not, and falls back to ordinal sample-letter order, which is
the same fallback the manuscript itself reports using for that specific arm.


In [ ]:
kent_specs = [
    ('S1A_pop_17978_TGC',   'TGC_17978',  'A. baumannii ATCC17978', 'Tigecycline', 'TGC'),
    ('S1B_pop_BAA747_TGC',  'TGC_BAA747', 'A. baumannii BAA-747',   'Tigecycline', 'TGC'),
    ('S3A_pop_17978_COL',   'COL_17978',  'A. baumannii ATCC17978', 'Colistin',    'COL'),
    ('S3B_pop_BAA747_COL',  'COL_BAA747', 'A. baumannii BAA-747',   'Colistin',    'COL'),
]

kent_class = {}
for sheet, run_label, org, drug, panel_key in kent_specs:
    df = load_kent_run(sheet, run_label, organism=org, drug=drug)
    panel = GENE_PANELS[('Kent2025_AAC', panel_key)]
    cls = classify_reactor(df, threshold=0.05, target_genes=panel['TARGET'], reg_genes=panel['REG'])
    kent_class[run_label] = cls
    print(f'--- {run_label} (n={len(cls)} reactors) ---')
    print(cls.sort_values('reactor').to_string(index=False))
    print(cls['ordering'].value_counts().to_dict())
    print()

# CIP reference arm shared by every comparison below: CAB-1 (A. baumannii),
# 0/5 REG-first, all 5 TARGET-first (Table 1)
cab1_n, cab1_reg = 5, 0
c, d = cab1_reg, cab1_n - cab1_reg

def reg_vs_not(cls_df):
    a = (cls_df['ordering'] == 'REG_FIRST').sum()
    return a, len(cls_df) - a

print('##### Table 6: drug-identity contrasts vs. CAB-1 (CIP) reference #####\n')

a, b = reg_vs_not(kent_class['COL_17978'])
print('COL (ATCC17978, strain-matched):', a, '/', a + b, 'REG-first  vs CAB-1', c, '/', c + d)
print(fisher_or_ha(a, b, c, d), '  (manuscript: P=0.0152, OR=40.3)\n')

a_pool = (kent_class['COL_17978']['ordering'] == 'REG_FIRST').sum() + (kent_class['COL_BAA747']['ordering'] == 'REG_FIRST').sum()
b_pool = len(kent_class['COL_17978']) + len(kent_class['COL_BAA747']) - a_pool
print('COL (ATCC17978 + BAA-747 pooled):', a_pool, '/', a_pool + b_pool, 'REG-first  vs CAB-1', c, '/', c + d)
print(fisher_or_ha(a_pool, b_pool, c, d), '  (manuscript: P=0.0090, OR=29.9)\n')

a3, b3 = reg_vs_not(kent_class['COL_17978']); c3, d3 = reg_vs_not(kent_class['COL_BAA747'])
print('COL, ATCC17978 vs BAA-747 (strain-vs-strain, expect ns):', a3, '/', a3+b3, ' vs ', c3, '/', c3+d3)
print(fisher_or_ha(a3, b3, c3, d3), '  (manuscript: P=1.0)\n')

a4, b4 = reg_vs_not(kent_class['TGC_17978'])
print('TGC (ATCC17978, strain-matched):', a4, '/', a4 + b4, 'REG-first  vs CAB-1', c, '/', c + d)
print(fisher_or_ha(a4, b4, c, d), '  (manuscript: P=0.0022, OR=143.0)\n')

a_pool2 = (kent_class['TGC_17978']['ordering'] == 'REG_FIRST').sum() + (kent_class['TGC_BAA747']['ordering'] == 'REG_FIRST').sum()
b_pool2 = len(kent_class['TGC_17978']) + len(kent_class['TGC_BAA747']) - a_pool2
print('TGC (ATCC17978 + BAA-747 pooled):', a_pool2, '/', a_pool2 + b_pool2, 'REG-first  vs CAB-1', c, '/', c + d)
print(fisher_or_ha(a_pool2, b_pool2, c, d), '  (manuscript: P=0.00023, OR=253.0)\n')

a6, b6 = reg_vs_not(kent_class['TGC_17978']); c6, d6 = reg_vs_not(kent_class['TGC_BAA747'])
print('TGC, ATCC17978 vs BAA-747 (strain-vs-strain, expect ns):', a6, '/', a6+b6, ' vs ', c6, '/', c6+d6)
print(fisher_or_ha(a6, b6, c6, d6), '  (manuscript: P=1.0)')


## 7. Ampicillin selection-strength comparison (Cisneros-Mayoral et al. 2022, *Mol Biol Evol*) — Table 2

**Update:** the journal's own supplementary file (`msac185_supplementary_data.zip`) never contained more than
LaTeX source and an 8-page methods PDF (confirmed twice, including a re-upload of the same file). The authors
deposited the actual per-mutation frequency tables separately, in their own GitHub repository
(`ccg-esb-lab/evoAMP`) rather than through the journal. Two further wrinkles specific to this source, found by
checking against the raw files rather than assuming:

- The pinned Zenodo release (`v1`, DOI 10.5281/zenodo.7080457) — normally the more citable, stable choice —
  turns out to **predate** the genomic-analysis notebooks and the mutation tables they load. Both exist only
  on the live `main` branch. Anyone citing this repository's data availability should point to a specific
  commit or a newer tagged release, not `v1`, or the mutation tables this section depends on will be missing.
- There is no single file playing the role of Kent's or Zlamal's supplementary spreadsheet. The relevant data
  is split across `data/MS_filt_all_mut.csv` and `data/SS_filt_all_mut.csv` (population-level, high-frequency-
  filtered mutations, three phase-level snapshots per replicate rather than a continuous time series), which
  the repository's own `code/r-files/Figure3_CDE.ipynb` loads to produce mutation-type frequency plots — but
  that notebook does not itself perform any TARGET/REG pathway classification; that classification is this
  manuscript's own contribution, applied on top of the raw frequencies below.

Because each replicate is only sampled at three phase-level snapshots (not a continuous series), the
manuscript uses a different — coarser but unambiguous — decision rule here than the ≥5%-threshold,
earliest-crossing rule used for the continuous-time datasets elsewhere in this notebook: a gene counts as
"fixed" if its population frequency reaches 100% by the end of Phase 1. TARGET = {ftsI} (penicillin-binding
protein 3, the direct β-lactam target); REG = {acrR, clpX, lon, rpoD, marR, acrB} (efflux-regulatory and
efflux-structural loci), with phoQ tracked separately and reported as "REG-adjacent" per the manuscript's own
qualification (a two-component regulator upstream of, but not itself part of, the core acrAB-regulatory set).

Clone-level linkage (ftsI+phoQ, and a subset with an additional acrB mutation, within single sequenced
clones from MS Replicate 4) is also claimed in the manuscript text, sourced to the original study's
Supplementary Data Set S5. No clone-level genotype file (as opposed to the population-level tables used
below) was found anywhere in this repository, at either the `v1` tag or the current `main` branch, so that
specific claim remains unverified by this notebook — flagged again in Section 12.

### Open, non-blocking item from this session

- **A third gene also reaches 100% by end of Phase 1 in MS, not mentioned in Table 2.**
  `yoeA–insH7` (a ~35.6 kb IS-mediated deletion, MS Replicate 2) hits 100% frequency at Phase 1 and then
  drops to 0% at both Phase 2 and Phase 3 -- itself an odd trajectory for a supposedly fixed deletion in an
  asexual lineage, and plausibly a variant-calling artifact rather than genuine fixation-then-loss. It falls
  outside the ftsI/REG/phoQ scheme entirely, which is presumably why Table 2 doesn't mention it (the same
  kind of principled exclusion as `trm` in the Kent/TGC panel, Section 6), but that exclusion isn't stated
  explicitly the way it is for `trm`. Doesn't change Table 2's MS-vs-SS conclusion either way, but is exactly
  the kind of loose end worth resolving with an explicit sentence before submission, in case a reviewer
  reruns the numbers independently and asks about it.


In [ ]:
def load_cisneros_mutations(filename, regime_label):
    """
    Loads one of Cisneros-Mayoral et al.'s population-level, high-frequency-
    filtered mutation tables (as deposited in their GitHub repo, not the
    journal's own supplementary file -- see markdown above). Each row is one
    mutation in one replicate, with frequency at three phase-level snapshots
    rather than a continuous time series.

    Intergenic mutations are annotated as e.g. 'clpX \u2192 / \u2192 lon' (gene
    upstream / gene downstream of the variant). A naive strip of the arrow
    characters alone concatenates these into unmatchable strings like
    'clpXlon' -- caught by testing this function's output against the raw
    'gene' column before trusting it, not by inspection. Splitting on both
    arrows and the slash keeps each flanking gene as a separate, matchable
    token instead.
    """
    df = pd.read_csv(f'{UPLOAD_DIR}/{filename}', index_col=0)
    df['gene_parts'] = df['gene'].apply(lambda raw: [p.strip() for p in re.split(r'[\u2192\u2190/]', str(raw)) if p.strip()])
    df['gene_clean'] = df['gene_parts'].apply('/'.join)
    df['regime'] = regime_label
    return df[['position', 'mutation', 'gene', 'gene_parts', 'gene_clean', 'MutType',
               'Phase1', 'Phase2', 'Phase3', 'Replic', 'regime']]

TARGET_AMP = {'ftsI'}
REG_AMP = {'acrR', 'clpX', 'lon', 'rpoD', 'marR', 'acrB'}
REG_ADJACENT_AMP = {'phoQ'}  # tracked separately, per the manuscript's own "REG-adjacent" qualification

def classify_gene_parts(parts):
    if any(p in TARGET_AMP for p in parts):
        return 'TARGET'
    if any(p in REG_AMP for p in parts):
        return 'REG'
    if any(p in REG_ADJACENT_AMP for p in parts):
        return 'REG-adjacent'
    return 'unclassified'

ms = load_cisneros_mutations('MS_filt_all_mut.csv', 'MS')
ss = load_cisneros_mutations('SS_filt_all_mut.csv', 'SS')
ms_ss = pd.concat([ms, ss], ignore_index=True)

fixed_phase1 = ms_ss[ms_ss['Phase1'] == 100.0].copy()
fixed_phase1['pathway_class'] = fixed_phase1['gene_parts'].apply(classify_gene_parts)

print('--- Genes reaching 100% population frequency by end of Phase 1 ---\n')
for regime in ['MS', 'SS']:
    sub = fixed_phase1[fixed_phase1['regime'] == regime].sort_values(['gene_clean', 'Replic'])
    print(f'{regime}:')
    print(sub[['gene_clean', 'pathway_class', 'Replic', 'position', 'MutType']].to_string(index=False))
    print(' unique genes:', sorted(sub['gene_clean'].unique()))
    print()

# n replicates per regime, from the original transfer-tracking table (independent
# of the filtered mutation table, which only lists replicates carrying a
# high-frequency mutation and so can undercount)
reps = pd.read_csv(f'{UPLOAD_DIR}/AMP_transfer_reps.csv')
ms_reps = [c for c in reps.columns if c.startswith('MS_R')]
ss_reps = [c for c in reps.columns if c.startswith('SS_R')]
print(f'Replicates tracked in original experiment: MS={ms_reps}, SS={ss_reps}')
print('(manuscript: n=4 replicates per regime)')

print('\n##### Table 2 reproduction #####')
print('MS: TARGET/REG-adjacent genes fixed by Phase 1 ->',
      sorted(fixed_phase1.loc[fixed_phase1['regime'] == 'MS', 'gene_clean'].unique()),
      ' (manuscript: ftsI, phoQ)')
print('SS: REG-only genes fixed by Phase 1 ->',
      sorted(fixed_phase1.loc[fixed_phase1['regime'] == 'SS', 'gene_clean'].unique()),
      ' (manuscript: acrR, clpX/lon, rpoD)')
ftsI_in_ss = ss['gene_parts'].apply(lambda p: 'ftsI' in p).any()
print('ftsI detected anywhere in SS at any phase (any frequency, any replicate)?', ftsI_in_ss,
      ' (manuscript: "not detected above threshold in any replicate at any phase")')

unclassified = fixed_phase1[fixed_phase1['pathway_class'] == 'unclassified']
if len(unclassified):
    print()
    print("NOTE: gene(s) also reaching 100% by Phase 1 but NOT listed in the manuscript's")
    print('Table 2 (falls outside the TARGET/REG/REG-adjacent scheme, so silently excluded')
    print('rather than misclassified -- flagged here rather than assumed innocuous):')
    print(unclassified[['regime', 'gene_clean', 'Replic', 'mutation', 'MutType', 'Phase1', 'Phase2', 'Phase3']].to_string(index=False))



## 8. GP6 (Leyn et al. 2024, *NPJ Antimicrob Resist*) — Table 5

`media-1.xlsx` sheet `S1A` = E. coli population time-series (6 reactors), `S1B` = A. baumannii population
time-series (6 reactors), `S1C` = quality metrics only (not mutation data). `media-2.xlsx` (`S2A`/`S2B`/`S2C`)
are **clone-level** data (different schema entirely — selected-clone genotypes/MICs, not a reactor time
series), so they are not run through `classify_reactor()` here.

**Two bugs found and fixed against the raw data (see Section 3 / Section 8 loader for details):**
1. The original sheet→run mapping treated all six `S1A/B/C` + `S2A/B/C` sheets as six separate population
   runs. `S1C`/`S2A-C` are not population time series at all — corrected to just `S1A` (E. coli) and `S1B`
   (A. baumannii).
2. `media-1.xlsx` mixes SNP/indel rows with **CNV/coverage rows** (`Mutation` column `== '-'`, a block of
   ~55 genes including `mdtK` with values already >50% at t=0 and exceeding 1.0 later — a read-depth
   ratio, not a 0–1 frequency). Left in, these forced REG_FIRST at t=0 for every reactor. Now excluded in
   `load_leyn_gp6()`.


In [ ]:
gp6_pop_pairs = [
    (f'{UPLOAD_DIR}/media-1.xlsx', 'S1A', 'GP6_Ecoli'),
    (f'{UPLOAD_DIR}/media-1.xlsx', 'S1B', 'GP6_Abaumannii'),
]
gp6_df = load_leyn_gp6(gp6_pop_pairs)
gp6_df = gp6_df.dropna(subset=['time_h'])
print('GP6 tidy rows:', len(gp6_df))
print(gp6_df.groupby('run')['reactor'].nunique())

gp6_out = []
for run, g in gp6_df.groupby('run'):
    panel = GENE_PANELS[('Leyn2024_NPJ', run)]
    res = classify_reactor(g, threshold=0.05, target_genes=panel['TARGET'], reg_genes=panel['REG'])
    res['run'] = run
    gp6_out.append(res)
gp6_class = pd.concat(gp6_out, ignore_index=True)
print(gp6_class.sort_values(['run', 'reactor']).to_string(index=False))
print()
print(gp6_class.groupby(['run', 'ordering']).size().unstack(fill_value=0))

# --- Table 5 Fisher's exact tests: GP6 vs CIP, per organism ---
# CIP E. coli reference = CEC-2 + CEC-4 (10 reactors, all TARGET-first, Table 1)
cip_ecoli = zlamal_class[zlamal_class['run'].isin(['CEC-2', 'CEC-4'])]
a, b, c, d = build_2x2_reg_vs_not(gp6_class[gp6_class['run'] == 'GP6_Ecoli'], cip_ecoli)
print()
print('E. coli, GP6 vs CIP:  GP6 REG-first', a, '/', a + b, '  CIP REG-first', c, '/', c + d)
print(fisher_or_ha(a, b, c, d), '  (manuscript: p=0.0082, OR=37.8)')

# CIP A. baumannii reference = CAB-1 (5 reactors, all TARGET-first, Table 1)
cip_abau = zlamal_class[zlamal_class['run'] == 'CAB-1']
a, b, c, d = build_2x2_reg_vs_not(gp6_class[gp6_class['run'] == 'GP6_Abaumannii'], cip_abau)
print()
print('A. baumannii, GP6 vs CIP:  GP6 REG-first', a, '/', a + b, '  CIP REG-first', c, '/', c + d)
print(fisher_or_ha(a, b, c, d), '  (manuscript: p=0.0152, OR=40.3)')
print()
print('NOTE: for A. baumannii GP6 Reactor 6, this code reports TARGET_FIRST (target crosses at')
print('77h, REG at 123h); the manuscript narrative describes the 6th non-REG-first reactor as')
print('"co-emergent". This labeling difference does not change the REG-first count or either')
print('p-value/OR above (both reproduce exactly either way) -- flagged as an open, non-consequential')
print('discrepancy rather than silently resolved.')

## 9. Rifampicin / environmental-rate dependence (Lindsey et al. 2013, *Nature*)

The secondary line of evidence about rate-dependent accessibility of multi-locus genotypes. Supplementary
Table 2 (isolate-level mutation list across Gradual/Moderate/Sudden treatments) is transcribed directly
from the supplementary PDF text (`41586_2013_BFnature11879_MOESM27_ESM.pdf`), since it is stored there as
a text table rather than a machine-readable spreadsheet.

In [ ]:
lindsey = load_lindsey_table2()
print(lindsey.shape)
print(lindsey.groupby('treatment').size())

# Number of *distinct* rpoB mutations fixed per isolate, by treatment --
# a simple proxy for genotype complexity in this dataset (compare with
# mannwhitney(), in the same spirit as Section 5).
n_mut_per_isolate = lindsey.groupby(['treatment', 'population']).size().reset_index(name='n_mutations')
print(n_mut_per_isolate.groupby('treatment')['n_mutations'].describe())

gradual = n_mut_per_isolate.loc[n_mut_per_isolate['treatment'] == 'Gradual', 'n_mutations'].values
sudden = n_mut_per_isolate.loc[n_mut_per_isolate['treatment'] == 'Sudden', 'n_mutations'].values
print('Gradual vs Sudden, n mutations per isolate:', mannwhitney(gradual, sudden))


## 10. Triclosan / cadC-fabI linkage (Leyn et al., curated from Data Set S6)

Uses the already-tidied clone-level CSVs (`S6_cadC_fabI_linked_clones.csv`, `S6_clone_genotypes_MIC.csv`)
rather than re-parsing `000553_2.xlsx` from scratch, since these were previously curated by hand from
Data Set S6 of that paper.

In [ ]:
linked_clones, all_clones = load_leyn_triclosan_clones()
print('Linked-clone table:', linked_clones.shape)
print(linked_clones.head())
print()
print('Full clone genotype/MIC table:', all_clones.shape)
print(all_clones['Haplotype_note'].value_counts() if 'Haplotype_note' in all_clones.columns else all_clones.head())


## 11. Fleroxacin dose-response (Yu et al. 2025, *Microbiol Spectr*) — ESM Table S2

Implemented (was previously left as a TODO — the compound-string cell format below is exactly what
required hand-verification rather than an automated guess):

`spectrum.02981-24-s0002.xlsx` (Table S1, low dose, Cycles 1–3) and `spectrum.02981-24-s0003.xlsx`
(Table S2, high dose, Cycles 5–7). Cells store per-position mutation strings, e.g. `"13.77%(M243I)"`, or,
when multiple positions co-occur in the same gene, a `+`-joined list ending in an explicit pre-summed
total, e.g. `"3.04%(L151P)+3.94%(A30T)+3.86%(T39A)=15.11%"`. Per the sheets' own header note ("mutation
frequencies of different positions in the same gene were summed"), the parser takes that explicit
trailing total when present, and the single parsed percentage otherwise — it never re-sums the individual
terms itself. `spectrum.02981-24-s0004.xlsx` (Table S3) is clone-level data for the high-resistance
isolates (not a time series) and is not used here.

Threshold is 10% (not 5%) and the gene panel is `TARGET = {gyrA, gyrB}`, `REG = {nfxB, mexR, nalC, nalD}`,
both per ESM Table S2's stated protocol.

**Verified against ESM Table S2: reproduces all 6/6 sample-level classifications exactly**
(0/6 TARGET-first, 4/6 REG-first-or-REG-only, 2/6 co-emergent, matching the manuscript's summary and the
individual per-sample crossing times/genes in the ESM table).

In [ ]:
import re as _re

def load_yu_fleroxacin():
    def parse_cell(val):
        if not isinstance(val, str) or val.strip() == '':
            return np.nan
        val = val.strip()
        total_str = val.split('=')[-1].strip() if '=' in val else val
        m = _re.search(r'([\d.]+)\s*%', total_str)
        return float(m.group(1)) / 100.0 if m else np.nan

    def gene_name(label):
        m = _re.search(r'\(([^)]+)\)', str(label))
        return m.group(1) if m else str(label).strip()

    out = []
    specs = [
        (f'{UPLOAD_DIR}/spectrum.02981-24-s0002.xlsx', 'Table S1', 'low'),
        (f'{UPLOAD_DIR}/spectrum.02981-24-s0003.xlsx', 'Table S2', 'high'),
    ]
    for filepath, sheet, dose in specs:
        raw = pd.read_excel(filepath, sheet_name=sheet, header=None)
        header = raw.iloc[2]
        for i in range(3, len(raw)):
            gene_label = raw.iat[i, 0]
            if not isinstance(gene_label, str) or gene_label.strip() == '':
                continue
            gene = gene_name(gene_label)
            for j in range(1, raw.shape[1]):
                col_label = header.iat[j]
                if not isinstance(col_label, str):
                    continue
                m = _re.match(r'(Low|High)-dose_Sample(\d+)_Cycle(\d+)', col_label)
                if not m:
                    continue
                sample, cycle = int(m.group(2)), int(m.group(3))
                freq = parse_cell(raw.iat[i, j])
                if pd.isna(freq):
                    continue
                out.append({'gene': gene, 'reactor': f'{dose}_Sample{sample}', 'time_h': cycle,
                             'frequency': freq, 'dose': dose, 'study': 'Yu2025_MicrobiolSpectr',
                             'run': f'{dose}-dose'})
    return pd.DataFrame(out)


yu_df = load_yu_fleroxacin()
print(len(yu_df), 'rows,', yu_df['reactor'].nunique(), 'samples')

yu_out = []
for run, g in yu_df.groupby('run'):
    panel = GENE_PANELS[('Yu2025_MicrobiolSpectr', run)]
    for reactor, gr in g.groupby('reactor'):
        res = classify_reactor(gr, threshold=0.10, target_genes=panel['TARGET'], reg_genes=panel['REG'],
                                reactor_col='reactor')
        res['run'] = run
        yu_out.append(res)
yu_class = pd.concat(yu_out, ignore_index=True)
print(yu_class[['reactor', 'first_target_crossing_h', 'first_reg_crossing_h', 'ordering']]
      .sort_values('reactor').to_string(index=False))
print()
print(yu_class['ordering'].value_counts())
print('Manuscript / ESM Table S2: 0/6 TARGET-first, 4/6 REG-first-or-REG-only, 2/6 co-emergent')

## 12. Still not implemented / still missing data — flagged rather than guessed

- **Cisneros-Mayoral et al., clone-level co-occurrence claim (ampicillin, *Mol Biol Evol*)** — Table 2 itself
  is now reproduced in Section 7 from the authors' GitHub repository (`ccg-esb-lab/evoAMP`, `main` branch;
  the journal's own `msac185_supplementary_data.zip` never contained more than LaTeX source and a methods
  PDF, confirmed on two separate uploads of that file). What is **not** resolved is the clone-level linkage
  claim (ftsI+phoQ, and a subset with an additional acrB mutation, co-occurring within single sequenced
  clones from MS Replicate 4, sourced to the original study's Supplementary Data Set S5) — no clone-level
  genotype file was found in the repository at either the `v1` tag or the current `main` branch, only
  population-level tables. That specific sentence still rests on the manuscript's own reported numbers only.
- **`elife-47612-supp2.xlsx`** and **`msab025_supplementary_data.pdf`** — checked both; they are the same
  foreign study (erythromycin dose-response / AcrB-GFP efflux dynamics in *E. coli* AG100/TB108/eTB108,
  not colistin/tigecycline/CIP/rifampicin/ampicillin), not cited anywhere in this manuscript's reference
  list (1–16). No loader was written for either. If these belong to a different project, they can likely
  be left out of this notebook's data folder.

### Open, non-blocking item from this session (not a bug, needs your judgement)

- **Genotype complexity, CEC-2 vs CEC-4 (Section 5).** Re-running `load_zlamal_clone_complexity()` against
  the real Data Set S3 gives mean n_mutations = 2.29 (CEC-2, n=17) vs 2.14 (CEC-4, n=7); the manuscript's
  ESM reports 1.94 (n=17) vs 2.00. Sample sizes match but the means don't exactly, which suggests a
  difference in exactly which rows count as a "mutation" (e.g. whether all `+`-marked rows in Data Set S3
  are included, or only a subset such as "potential driver mutations"). The comparison is reported as
  non-significant either way (manuscript P=0.84), so this doesn't change any conclusion, but the loader
  should not be treated as an exact reproduction of this specific number without checking the driver-vs-
  passenger row categories in Data Set S3 against your Methods description.

## 13. Data availability / provenance / status

| Study | File(s) used here | Source | Status this session |
|---|---|---|---|
| Zlamal et al. 2021, *mBio* (CIP) | `mbio.00987-21-sd002.xlsx`, `-sd003.xlsx`, `-t0001.pdf` | Data Set S2, S3, Table 1 | **Table 1 reproduced exactly**, all 5 runs |
| Leyn et al. 2024, *NPJ Antimicrob Resist* (GP6) | `media-1.xlsx` (`S1A`,`S1B`) | Supplementary Table S1AB | **Table 5 reproduced exactly**, both organisms |
| Yu et al. 2025, *Microbiol Spectr* (fleroxacin) | `spectrum.02981-24-s0002.xlsx`, `-s0003.xlsx` | Supplementary Tables S1-S2 | **ESM Table S2 reproduced exactly**, 6/6 samples |
| Lindsey et al. 2013, *Nature* (rifampicin) | `41586_2013_BFnature11879_MOESM27_ESM.pdf` | Supplementary Table 2 | **Transcription verified exact**, 121/121 rows |
| Leyn et al. (triclosan) | `000553_2.xlsx`, curated `S6_*.csv` | Data Set S6 | Spot-checked against raw sheet, consistent |
| Kent et al. 2025, *Antimicrob Agents Chemother* (COL/TGC) | `aac.00809-25-s0002.xlsx` | Supplementary Tables S1-S14 | **Table 6 reproduced exactly**, all 6 sub-comparisons (17978, BAA-747, pooled, strain-vs-strain, both drugs) |
| Cisneros-Mayoral et al. 2022, *Mol Biol Evol* (ampicillin) | `MS_filt_all_mut.csv`, `SS_filt_all_mut.csv`, `AMP_transfer_reps.csv` | `github.com/ccg-esb-lab/evoAMP` (`main` branch, not the journal's own supplementary file) | **Table 2 reproduced exactly** (population-level); clone-level co-occurrence claim (Data Set S5) still unverified — no clone-level file found in the repo |
| `spectrum.02981-24-s0004.xlsx` (Yu et al. Table S3) | clone-level, not a time series | Supplementary Table S3 | Not used (not needed for the earliest-crossing classification) |
| `elife-47612-supp2.xlsx` | — | — | **Not cited in this manuscript's reference list** — no loader written |
| `msab025_supplementary_data.pdf` | — | — | Same foreign study as `elife-47612-supp2.xlsx` (erythromycin/AcrB-GFP) — **not cited**, no loader written |

None of the underlying raw data are redistributed in this notebook or repository — only the loader code
that reads locally-placed copies of the original authors' publicly deposited supplementary files. When
depositing this notebook (e.g. to Zenodo / GitHub) for the manuscript's Data, Code and Materials
statement, link back to each original publication's own supplementary-material page rather than
re-hosting these files, in line with standard practice and each publisher's terms of reuse.